In [1]:
import os
import numpy as np
import pandas as pd
import scanpy as sc

import torch
from torch import nn
import torchmetrics
from importlib import reload
import pytorch_lightning as pl
from pytorch_lightning import callbacks 

from src import PATH
from src import _helper_net, _linear

In [2]:
from torch.utils.data import DataLoader, Dataset, TensorDataset

In [3]:
adata = sc.read_h5ad("/home/wergillius/Project/diffuse_differentiate/data/Barcodelet/integrated_mesc_group0_Nov7.h5ad")
adata

AnnData object with n_obs × n_vars = 5745 × 2000
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'RA', 'Wnt', 'TgfB', 'Bmp', 'Fgf', 'Notch', 'Shh', 'assignment', 'starting.state', 'dataset', 'control', 'cell_type', 'condition', 'dose_val', 'batch', 'dpt_groups', 'dpt_order', 'dpt_order_indices', 'dpt_pseudotime', 'RA+Wnt+Fgf', 'numeric_iloc', 'split', 'discrete_time'
    var: 'gene_name'
    uns: 'condition_colors', 'diff', 'diffmap_evals', 'dpt_changepoints', 'dpt_groups_colors', 'dpt_grouptips', 'iroot', 'neighbors', 'umap', 'unique_token_dict'
    obsm: 'Tr_SampledX_r100', 'X_diffmap', 'X_pca', 'X_umap'
    varm: 'PCs'
    obsp: 'connectivities', 'diff_connectivities', 'diff_distances', 'distances'

# Tensor ds

In [5]:
perturbations = ['RA', 'Wnt', 'TgfB', 'Bmp', 'Fgf', 'Notch', 'Shh']
assignment_C = adata.obs[perturbations].values
gene_X = adata.X

X =  np.concatenate([gene_X, assignment_C], axis=1)

Y = adata.obsm['Tr_SampledX_r100'].copy()

train_idx = adata.obs.split != 'test'
test_idx = adata.obs.split == 'test'

x_train = torch.from_numpy(X[train_idx]).float()
x_test = torch.from_numpy(X[test_idx]).float()

Y_train = torch.from_numpy(Y[train_idx]).float()
Y_test = torch.from_numpy(Y[test_idx]).float()

In [6]:
train_ds = TensorDataset(x_train, Y_train)
train_dl = DataLoader(train_ds, batch_size=32, num_workers=8, shuffle=True)

test_ds = TensorDataset(x_test, Y_test)
test_dl = DataLoader(test_ds, batch_size=32,  num_workers=8, shuffle=False)

In [7]:
x_train.shape

torch.Size([5170, 2007])

# model

In [8]:
reload(_helper_net)

<module 'src._helper_net' from '/home/wergillius/Project/diffuse_differentiate/src/_helper_net.py'>

In [10]:
from scipy import stats

def evaluate_r(test_Y, pred_Y):

    r_dict = []
    for i, gene in enumerate(adata.var.index):
        r_dict.append(
            {"gene":gene, 'r2': stats.pearsonr(test_Y[:,i], pred_Y[:,i])[0]**2}
            # {"gene":gene, 'r2': stats.spearmanr(test_Y[:,i], pred_Y[:,i])[0]**2}
        )

    return pd.json_normalize(r_dict)

# Training linear model

In [20]:
L_m = _linear.Linear_model(2007, 2000, 3e-4,1e-3)
Trainer = pl.Trainer(accelerator='gpu', gpus=[0],
                     default_root_dir = os.path.join(PATH.pth_dir, 'Barcodelet', 'Linear_model'),
                     callbacks=[
                         callbacks.ModelCheckpoint(save_top_k=2, monitor="val_loss"),
                         callbacks.EarlyStopping(monitor="val_loss", mode="min", patience=10)
                         ],
                    )
Trainer.fit(L_m, train_dataloaders=train_dl, 
            val_dataloaders=test_dl)

### evaluating Linear model on ytest

In [28]:
L_m = L_m.eval().to('cpu');

Y_pred_LM = []
with torch.no_grad():
    for X,Y in test_dl:
        Y_pred_LM.append( L_m(X) )

In [38]:
ypred = torch.concat(Y_pred_LM)
rdf = evaluate_r(ypred, Y_test)

/home/wergillius/.conda/envs/GenDiff/lib/python3.8/site-packages/scipy/stats/_stats_py.py:4427: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  warnings.warn(stats.ConstantInputWarning(msg))


In [39]:
rdf

,gene,r2
0,ENSMUSG00000031502,0.071300
1,ENSMUSG00000000031,0.422740
2,ENSMUSG00000038793,0.335346
3,ENSMUSG00000031503,0.109227
4,ENSMUSG00000010760,0.403993
...,...,...
1995,ENSMUSG00000006931,0.413602
1996,ENSMUSG00000034574,0.521444
1997,ENSMUSG00000040483,0.613762
1998,ENSMUSG00000026288,0.510835


In [31]:
rdf.to_csv("/home/wergillius/Project/diffuse_differentiate/result/Barcodelet/LinearModel_performance.csv")

# training CAE model

## shallow

In [39]:
VAE = _linear.CAE_model(2007, 2000, hidden=[2048],
         lr=1e-4,weight_decay=1e-4)

Trainer = pl.Trainer(accelerator='gpu', gpus=[0], auto_lr_find=True,
                     default_root_dir = os.path.join(PATH.pth_dir, 'Barcodelet', 'CVAE_model'),
                     callbacks=[
                         callbacks.ModelCheckpoint(save_top_k=2, monitor="val_loss"),
                         callbacks.EarlyStopping(monitor="val_loss", mode="min", patience=10)
                         ],
                    )

Trainer.fit(VAE, train_dataloaders=train_dl, 
            val_dataloaders=test_dl)

In [22]:
ckpt="/home/wergillius/data/diffuse_differentiate/Barcodelet/CVAE_model/lightning_logs/version_2/checkpoints/epoch=23-step=3888.ckpt"

In [23]:
CAE = _linear.CAE_model.load_from_checkpoint(ckpt)

In [24]:
CAE = CAE.eval().to('cpu');

Y_pred_CAE = []
with torch.no_grad():
    for X,Y in test_dl:
        Y_pred_CAE.append( CAE(X) )

In [25]:
ypred_CAE = torch.concat(Y_pred_CAE)
rdf_CAE = evaluate_r(ypred_CAE, Y_test)

/home/wergillius/.conda/envs/GenDiff/lib/python3.8/site-packages/scipy/stats/_stats_py.py:4427: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  warnings.warn(stats.ConstantInputWarning(msg))


In [27]:
rdf_CAE.describe()

,r2
count,1.984000e+03
mean,2.815494e-01
std,1.126464e-01
min,1.518739e-08
25%,2.437880e-01
50%,3.046999e-01
75%,3.533788e-01
max,6.077800e-01


In [19]:
rdf_CAE.to_csv("/home/wergillius/Project/diffuse_differentiate/result/Barcodelet/CAE_performance.csv")

## deep  CAE

In [8]:
encoder_kwargs = {'dimensions':[2007, 1024, 1024],
                    "skip_connection": True,
                  "output_activation":"ReLU"}

decoder_kwargs = {'dimensions':[1024, 1024, 2000], 
                "skip_connection": True,
                  "use_batchnorm":False, 
                  "output_activation":None}


CAE = _linear.CAE_model(
              encoder_kwargs = encoder_kwargs, 
              decoder_kwargs = decoder_kwargs,
              lr=1e-5,
              weight_decay=1e-9)

In [10]:
Trainer = pl.Trainer(accelerator='gpu', gpus=[0], auto_lr_find=True,
                     default_root_dir = os.path.join(PATH.pth_dir, 'Barcodelet', 'CVAE_deep_model'),
                     callbacks=[
                         callbacks.ModelCheckpoint(save_top_k=2, monitor="val_loss"),
                         callbacks.EarlyStopping(monitor="val_loss", mode="min", patience=10)
                         ],
                    )


/home/wergillius/.conda/envs/GenDiff/lib/python3.8/site-packages/pytorch_lightning/trainer/connectors/accelerator_connector.py:467: LightningDeprecationWarning: Setting `Trainer(gpus=[0])` is deprecated in v1.7 and will be removed in v2.0. Please use `Trainer(accelerator='gpu', devices=[0])` instead.
  rank_zero_deprecation(
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


In [11]:
Trainer.fit(CAE, train_dataloaders=train_dl,
            val_dataloaders=test_dl)

/home/wergillius/.conda/envs/GenDiff/lib/python3.8/site-packages/pytorch_lightning/loops/utilities.py:94: PossibleUserWarning: `max_epochs` was not set. Setting it to 1000 epochs. To train without an epoch limit, set `max_epochs=-1`.
  rank_zero_warn(
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]

  | Name           | Type    | Params
-------------------------------------------
0 | loss_fn        | MSELoss | 0     
1 | train_r2_score | R2Score | 0     
2 | val_r2_score   | R2Score | 0     
3 | encoder        | MLP     | 3.1 M 
4 | decoder        | MLP     | 3.1 M 
-------------------------------------------
6.2 M     Trainable params
0         Non-trainable params
6.2 M     Total params
24.822    Total estimated model params size (MB)


Sanity Checking: 0it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

...

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

# training latent-CAE model

In [10]:
reload(_linear)

<module 'src._linear' from '/home/wergillius/Project/diffuse_differentiate/src/_linear.py'>

In [28]:
LCAE = _linear.LatentAdd_CAE(2007, 7, 2000, hidden=[2048],
         lr=1e-4,weight_decay=1e-20)

In [ ]:
Trainer = pl.Trainer(accelerator='gpu', gpus=[0], auto_lr_find=True,
                     default_root_dir = os.path.join(PATH.pth_dir, 'Barcodelet', 'LCAE_model'),
                     callbacks=[
                         callbacks.ModelCheckpoint(save_top_k=2, monitor="val_loss"),
                         callbacks.EarlyStopping(monitor="val_loss", mode="min", patience=10)
                         ],
                    )

Trainer.fit(LCAE, train_dataloaders=train_dl, 
            val_dataloaders=test_dl)

## evaluate LCAE

In [11]:
ckpt="/home/wergillius/data/diffuse_differentiate/Barcodelet/LCAE_model/lightning_logs/version_6/checkpoints/epoch=24-step=4050.ckpt"

In [12]:
LCAE = _linear.LatentAdd_CAE.load_from_checkpoint(ckpt)

In [13]:
LCAE = LCAE.eval().to('cpu');

Y_pred_LCAE = []
with torch.no_grad():
    for X,Y in test_dl:
        Y_pred_LCAE.append( LCAE(X) )

In [14]:
ypred_LCAE = torch.concat(Y_pred_LCAE)
rdf_LCAE = evaluate_r(ypred_LCAE, Y_test)

/home/wergillius/.conda/envs/GenDiff/lib/python3.8/site-packages/scipy/stats/_stats_py.py:4427: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  warnings.warn(stats.ConstantInputWarning(msg))


In [15]:
rdf_LCAE.describe()

,r2
count,1984.000000
mean,0.290616
std,0.109654
min,0.000009
25%,0.256112
50%,0.312101
75%,0.357786
max,0.632537


In [85]:
rdf_CAE.to_csv("/home/wergillius/Project/diffuse_differentiate/result/Barcodelet/LCAE_performance.csv")